# Temporal Deep Learning for Crop Yield Prediction Using Real Data
**B.Tech Project | Google Colab Ready**


In [ ]:
!pip -q install pandas numpy requests scikit-learn tensorflow matplotlib
import os, zipfile, glob, time, requests, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
warnings.filterwarnings('ignore')
np.random.seed(42); tf.random.set_seed(42)


## 1. Download official FAOSTAT crop-yield data

In [ ]:
URL='https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip'
ZIP='faostat_crops.zip'; OUT='faostat_data'
if not os.path.exists(ZIP):
    r=requests.get(URL,timeout=180); r.raise_for_status(); open(ZIP,'wb').write(r.content)
os.makedirs(OUT,exist_ok=True)
with zipfile.ZipFile(ZIP) as z: z.extractall(OUT)
files=glob.glob(OUT+'/**/*.csv',recursive=True)


In [ ]:
crop_file=next(f for f in files if 'All_Data' in os.path.basename(f))
raw=pd.read_csv(crop_file,encoding='latin-1')
india=raw[raw['Area'].eq('India')].copy(); major=['Wheat','Rice','Maize']
d=india[india['Item'].isin(major) & india['Element'].astype(str).str.lower().str.contains('yield')].copy()
d=d.rename(columns={'Year':'year','Item':'crop','Value':'yield_raw','Unit':'yield_unit'})[['year','crop','yield_raw','yield_unit']]
d['year']=pd.to_numeric(d['year'],errors='coerce'); d['yield_raw']=pd.to_numeric(d['yield_raw'],errors='coerce'); d=d.dropna()
d['yield_tonnes_per_ha']=np.where(d['yield_unit'].astype(str).str.contains('hg/ha',case=False,na=False),d['yield_raw']/10000,d['yield_raw'])
d=d.sort_values(['crop','year']).reset_index(drop=True)


## 2. Download real environmental data from NASA POWER

In [ ]:
LAT,LON=22.9734,78.6569; start_year,end_year=int(d.year.min()),int(d.year.max())
parameters='T2M,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN'; API='https://power.larc.nasa.gov/api/temporal/daily/point'
frames=[]
for year in range(start_year,end_year+1):
    try:
        q={'parameters':parameters,'community':'AG','longitude':LON,'latitude':LAT,'start':f'{year}0101','end':f'{year}1231','format':'JSON'}
        p=requests.get(API,params=q,timeout=180).json().get('properties',{}).get('parameter')
        if p:
            z=pd.DataFrame(p); z.index=pd.to_datetime(z.index.astype(str),format='%Y%m%d',errors='coerce'); frames.append(z[z.index.notna()].replace(-999,np.nan))
    except Exception as e: print('Skipped',year,e)
if not frames: raise RuntimeError('No NASA POWER data downloaded')
w=pd.concat(frames).apply(pd.to_numeric,errors='coerce').sort_index()
a=w.resample('YE').agg({'T2M':'mean','PRECTOTCORR':'sum','RH2M':'mean','ALLSKY_SFC_SW_DWN':'mean'}).reset_index()
a['year']=a['index'].dt.year; a=a.drop(columns='index').dropna()


## 3. Merge real data and create temporal sequences

In [ ]:
df=d[['year','crop','yield_tonnes_per_ha']].merge(a,on='year',how='inner')
df=pd.get_dummies(df,columns=['crop'],dtype=int).sort_values(['year']).reset_index(drop=True)
features=['T2M','PRECTOTCORR','RH2M','ALLSKY_SFC_SW_DWN']+[c for c in df.columns if c.startswith('crop_')]
WINDOW=5; X=[]; y=[]; yrs=[]
for cc in [c for c in df.columns if c.startswith('crop_')]:
    p=df[df[cc]==1].sort_values('year').reset_index(drop=True)
    for i in range(WINDOW,len(p)):
        X.append(p.loc[i-WINDOW:i-1,features].to_numpy(dtype=float))
        y.append(float(p.loc[i,'yield_tonnes_per_ha'])); yrs.append(int(p.loc[i,'year']))
X=np.asarray(X,dtype=np.float32); y=np.asarray(y,dtype=np.float32); yrs=np.asarray(yrs)
if len(X)<10: raise RuntimeError(f'Only {len(X)} temporal samples were created. Reduce WINDOW or check downloaded data.')
unique_years=np.unique(yrs); split_year=unique_years[max(1,int(len(unique_years)*0.8))-1]
tr=yrs<=split_year; te=~tr
if te.sum()==0: tr=np.arange(len(X))<max(1,int(len(X)*0.8)); te=~tr
sc=StandardScaler(); n_train=X[tr].shape[0]; steps=X.shape[1]; nf=X.shape[2]
Xtr=sc.fit_transform(X[tr].reshape(-1,nf)).reshape(n_train,steps,nf)
Xte=sc.transform(X[te].reshape(-1,nf)).reshape(X[te].shape[0],steps,nf)
ytr,yte=y[tr],y[te]
print('X train:',Xtr.shape,'X test:',Xte.shape,'Train samples:',len(ytr),'Test samples:',len(yte))


## 4. Train LSTM and evaluate

In [ ]:
# Safety checks for Colab and small real-world datasets
if len(Xtr) < 2 or len(Xte) < 1:
    raise RuntimeError(f'Insufficient data after split: train={len(Xtr)}, test={len(Xte)}')

tf.keras.backend.clear_session()
model=Sequential([
    Input(shape=(Xtr.shape[1],Xtr.shape[2])),
    LSTM(32,return_sequences=False),
    Dropout(0.20),
    Dense(16,activation='relu'),
    Dense(1)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),loss='mse',metrics=['mae'])

# validation_split can fail or create an unusable validation set on small datasets
use_validation=len(Xtr)>=10
callbacks=[EarlyStopping(monitor='val_loss' if use_validation else 'loss',patience=15,restore_best_weights=True)]
fit_kwargs={'epochs':150,'batch_size':min(16,len(Xtr)),'verbose':1,'callbacks':callbacks,'shuffle=False}
if use_validation: fit_kwargs['validation_split']=0.2
history=model.fit(Xtr,ytr,**fit_kwargs)

pred=model.predict(Xte,verbose=0).reshape(-1)
pred=np.nan_to_num(pred,nan=float(np.mean(ytr)),posinf=float(np.max(ytr)),neginf=float(np.min(ytr)))

mae=mean_absolute_error(yte,pred)
rmse=float(np.sqrt(mean_squared_error(yte,pred)))
r2=r2_score(yte,pred) if len(yte)>=2 else float('nan')
print(f'MAE: {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R2: {r2:.4f}' if np.isfinite(r2) else 'R2: Not available (requires at least 2 test samples)')

results=pd.DataFrame({'year':yrs[te],'actual_yield':yte,'predicted_yield':pred})
display(results)
results.to_csv('lstm_predictions.csv',index=False)

plt.figure(figsize=(9,4))
plt.plot(results['year'],results['actual_yield'],marker='o',label='Actual')
plt.plot(results['year'],results['predicted_yield'],marker='x',label='Predicted')
plt.xlabel('Year'); plt.ylabel('Yield (tonnes/hectare)'); plt.title('LSTM: Actual vs Predicted Crop Yield')
plt.grid(); plt.legend(); plt.tight_layout(); plt.show()

model.save('crop_yield_lstm.keras')
print('Saved model: crop_yield_lstm.keras')
